# 재정렬기를 공동인용으로 미세조정 (Colab / RunPod)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndigkwk/arxiv_graph_RAG/blob/main/notebooks/rerank_finetune_colab.ipynb)

위 배지를 누르면 이 노트북이 Colab에서 바로 열립니다. **런타임 > 런타임 유형 변경**에서
GPU(L4 또는 A100)를 고르고 위에서 아래로 실행하면 됩니다. 고칠 셀은 **1절 하나**입니다.

## 넘어야 하는 바 — 돌리기 전에 적어 둡니다

[ADR-057](https://github.com/rndigkwk/arxiv_graph_RAG/blob/main/docs/adr/ADR-057-fine-tune-the-reranker-on-cocitation.md)이
정한 선입니다. **여기 적어 두는 이유는 결과를 보고 바를 움직일 수 없게 하는 것**입니다 —
ADR-039·050에서 그렇게 해서 판정이 성립했습니다.

미래 라벨 과제의 **간선 0개 집단(1,112 시드), 후보 풀 50**에서 코사인을 **쌍체 구간이 0을
제외하도록** 이겨야 합니다.

| 지표 | 코사인 | 사전학습 재정렬기 | 미세조정이 넘어야 하는 선 |
|---|---:|---:|---|
| recall@5 | **0.1227** | 0.0868 | > 0.1227, 구간 0 제외 |
| hit@5 | **0.4739** | 0.3723 | > 0.4739, 구간 0 제외 |
| MRR | **0.3457** | 0.2523 | > 0.3457, 구간 0 제외 |

- 사전학습 판은 recall@5에서 **0.0359 낮습니다.** 그것을 메우고 **그 위로** 가야 합니다
  (41% 상대 개선). **동점은 통과가 아닙니다** — 코사인이 공짜입니다.
- 풀 50의 **상한이 0.2690**입니다. recall@5가 그보다 크게 나오면 **측정이 틀린 것**입니다.
- 간선 1~4개 집단(251 시드)을 깨뜨리면 전체로는 쓸 수 없습니다.

**판정은 이 노트북에서 하지 않습니다.** 여기서는 점수만 내고, 순위와 구간은 저장소가 냅니다 —
동점 처리까지 다른 변형과 같아야 비교가 성립합니다.

## 무엇을 학습하는가 — 추론이 보는 것과 같게

저장소가 보낸 학습 쌍은 **시드마다 `풀 ∩ 공동인용`을 1, 나머지 풀을 0**으로 둡니다.
풀 밖 양성은 들어 있지 않습니다 — 추론에서 볼 일이 없는 쌍이기 때문입니다(실측: 풀 49편에
양성 p50 **2**개, 비율 약 1:12).

그래프를 **2026-07**로 잘라 만들었습니다. 평가는 2026-08로 자르므로 **학습 라벨과 평가
정답이 구조적으로 분리**됩니다(내보내기 매니페스트의 `leak_check`가 0을 적어 둡니다).

## 1. 설정 — 여기만 바꿉니다

In [ ]:
# 이 셀만 바꾸고 나머지는 그대로 실행하세요.

# **이 모델이어야 합니다.** 같은 계열 셋 중 1024를 받는 것은 v2-m3뿐입니다
# (2026-10-20 실측, AutoConfig.max_position_embeddings):
#   bge-reranker-base    512   <- 1024를 주면 장치 어서션으로 터집니다
#   bge-reranker-large   512   <- 큰데도 512입니다
#   bge-reranker-v2-m3  8192   <- 이것
MODEL = "BAAI/bge-reranker-v2-m3"

# 쌍 길이. 이 말뭉치 실측은 중앙값 636, p95 850, 최대 1025였습니다.
#   512로 자르면 쌍의 84.8%가 잘립니다 -> 재는 것이 모델이 아니라 자른 자리가 됩니다
MAX_SEQ = 1024

# **학습은 채점보다 무겁습니다.** 역전파가 활성값을 들고 있어야 하므로 같은 배치에서
# VRAM이 몇 배입니다. 16으로 시작하고 첫 조각의 피크를 보고 올리세요 - 어림하지 말고
# 측정으로. OOM이 나면 자동으로 절반씩 줄입니다.
BATCH = 16

# 1에폭부터 봅니다. ADR-057이 적은 선: **한 에폭 뒤 손실이 내려가지 않으면 그것이 답이고,
# 더 돌리지 않습니다.** 늘리는 것은 결과를 보고 정합니다.
EPOCHS = 1
LR = 2e-5
WARMUP_RATIO = 0.1

# 검증에 뗄 시드 비율. 손실이 내려가는지 **학습 중에** 보려면 필요합니다.
VAL_RATIO = 0.1

# 저장소가 보낸 파일 넷. **HF 데이터셋에 올라가 있으면 그대로 받아 옵니다.**
TRAIN_PAIRS = "rerank_training_pairs.jsonl.gz"
TRAIN_TEXTS = "rerank_training_texts.jsonl.gz"
EVAL_PAIRS = "rerank_pairs.jsonl.gz"
EVAL_TEXTS = "rerank_texts.jsonl.gz"

USE_HF = True
DATA_REPO = "arxiv-graphrag-colab"

# 학습한 가중치도 HF에 올릴지. 끄면 점수만 올립니다(파일이 2.3GB입니다).
UPLOAD_WEIGHTS = False

# SMOKE=True면 시드 40개로 학습하고 40개만 채점해 코드가 도는지 봅니다.
SMOKE = False

## 2. 환경 확인

In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None or not torch.cuda.is_available():
    print('GPU가 없습니다. 런타임 > 런타임 유형 변경에서 GPU를 고르세요.')
    if not SMOKE:
        raise SystemExit('GPU 없이 학습을 시작하지 않습니다.')
else:
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'{name}  {total:.1f}GB')

    # **CUDA가 이미 망가져 있지 않은지 먼저 봅니다.** 장치 어서션(cudaErrorAssert)이 한 번
    # 나면 그 프로세스의 CUDA 컨텍스트가 끝까지 망가집니다. 그 뒤의 모든 CUDA 호출이
    # 실패하는데 보통 모델 로드에서 터져서 **원인처럼 보입니다** - 2026-10-20에 정확히
    # 그 일이 있었습니다. 작은 연산 하나로 몇 초 만에 가려냅니다.
    try:
        torch.zeros(8, device='cuda').sum().item()
        print('CUDA 상태 정상')
    except Exception as exc:
        raise SystemExit(
            f'CUDA가 이미 망가진 상태입니다({type(exc).__name__}). **런타임 > 세션 다시 '
            f'시작**을 누르고 1절부터 다시 돌리세요. 원문: {str(exc).splitlines()[0][:150]}'
        ) from None

    print(f'배치 {BATCH}, 최대 길이 {MAX_SEQ}로 시작합니다. OOM이 나면 절반씩 줄입니다.')

In [ ]:
%pip -q install -U sentence-transformers

# **설치가 됐는지 여기서 확인합니다.** pip는 의존성이 어긋나도 경고만 하고 넘어갑니다 -
# 2026-10-20에 huggingface_hub를 올렸다가 sentence-transformers와 어긋나 학습 직전에
# 터졌습니다. 임포트가 되는지 지금 보고, 안 되면 무엇을 해야 하는지 적습니다.
try:
    import sentence_transformers
    from sentence_transformers import CrossEncoder, InputExample
    print('sentence-transformers', sentence_transformers.__version__)
except Exception as exc:
    raise SystemExit(
        f'설치가 어긋났습니다({type(exc).__name__}: {exc}). **런타임 > 세션 다시 시작**을 '
        '누른 뒤 1절부터 다시 돌리세요.'
    ) from None

### 길이 한도 확인 — 가중치를 받기 전에 막습니다

In [ ]:
from transformers import AutoConfig

# 설정만 읽습니다(몇 KB). `MAX_SEQ`가 한도를 넘으면 2.3GB를 받을 이유가 없습니다 -
# 넘긴 채 돌리면 장치 어서션으로 죽고 **오류에 길이 이야기가 한 줄도 없습니다.**
_cfg = AutoConfig.from_pretrained(MODEL)
_limit = getattr(_cfg, 'max_position_embeddings', None)
print(f'{MODEL}: max_position_embeddings = {_limit}')
if _limit and MAX_SEQ > _limit - 2:
    raise SystemExit(
        f'MAX_SEQ={MAX_SEQ}가 이 모델의 한도({_limit})를 넘습니다. 1절에서 MAX_SEQ를 '
        f'{_limit - 2} 이하로 줄이거나 MODEL을 v2-m3로 두세요.'
    )

## 3. 허깅페이스 토큰 — **여기서 한 번만** 묻습니다

In [ ]:
import getpass

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print('Colab 시크릿의 HF_TOKEN을 씁니다.')
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF 토큰(write 권한 필요): ')

if not HF_TOKEN:
    raise SystemExit('토큰이 비었습니다. 4절의 받기와 7절의 올리기에 필요합니다.')

# 누구의 토큰인지만 확인합니다. 토큰 자체는 출력하지 않습니다.
from huggingface_hub import HfApi

HF_OWNER = HfApi(token=HF_TOKEN).whoami()['name']
print('계정:', HF_OWNER)

## 4. 쌍 받기 — 학습용과 평가용

**둘 다 받습니다.** 학습 쌍은 2026-07로 자른 그래프에서, 평가 쌍은 2026-08로 자른
그래프에서 나왔습니다. 섞이면 판정이 성립하지 않습니다.

In [ ]:
import gzip
import json
from pathlib import Path


def fetch(name):
    # 3절에서 받은 토큰을 재사용합니다. 여기서 다시 묻지 않습니다.
    path = Path(name)
    if path.exists():
        return path
    if not USE_HF:
        raise SystemExit(f'{name}이 없습니다. 왼쪽 파일 창에 올리거나 USE_HF=True로 두세요.')
    from huggingface_hub import hf_hub_download
    try:
        return Path(hf_hub_download(f'{HF_OWNER}/{DATA_REPO}', name,
                                    repo_type='dataset', token=HF_TOKEN))
    except Exception as exc:
        # **404를 트레이스백으로 두지 않습니다.** 없는 이유는 거의 항상 "아직 올리지
        # 않았다"이고, 그때 필요한 것은 예외 11겹이 아니라 **다음에 칠 명령**입니다.
        if 'EntryNotFound' not in type(exc).__name__ and '404' not in str(exc):
            raise
        raise SystemExit(
            f'''{name}이 HF 데이터셋({HF_OWNER}/{DATA_REPO})에 없습니다.
저장소에서 만들어 올리세요 — 학습 쌍 둘만 올리면 됩니다
(평가 쌍 rerank_pairs/texts는 이미 올라가 있을 수 있습니다):

  uv run python -m evals.export_rerank_training --max-seeds 1000
  hf upload {HF_OWNER}/{DATA_REPO} \
      exports/colab/{name} {name} --repo-type dataset
'''
        ) from None


def read_jsonl(path):
    with gzip.open(path, 'rt', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle]


train_rows = read_jsonl(fetch(TRAIN_PAIRS))
train_texts = {r['arxiv_id']: r['text'] for r in read_jsonl(fetch(TRAIN_TEXTS))}
eval_rows = read_jsonl(fetch(EVAL_PAIRS))
eval_texts = {r['arxiv_id']: r['text'] for r in read_jsonl(fetch(EVAL_TEXTS))}

if SMOKE:
    train_rows, eval_rows = train_rows[:40], eval_rows[:40]

# 텍스트가 없으면 멈춥니다. 빈 문자열로 학습하거나 채점하면 그것이 모델의 사실처럼
# 보고서에 들어갑니다.
for label, rows, texts, keys in (
    ('학습', train_rows, train_texts, ('positives', 'negatives')),
    ('평가', eval_rows, eval_texts, ('candidates',)),
):
    missing = {p for row in rows
               for p in [row['seed'], *(x for k in keys for x in row[k])]
               if p not in texts}
    if missing:
        raise SystemExit(f'{label} 텍스트가 없는 논문 {len(missing)}편: {sorted(missing)[:5]}')

n_pos = sum(len(r['positives']) for r in train_rows)
n_neg = sum(len(r['negatives']) for r in train_rows)
print(f'학습 시드 {len(train_rows)}개 / 양성 {n_pos} / 음성 {n_neg}  (1:{n_neg / max(1, n_pos):.1f})')
print(f'평가 시드 {len(eval_rows)}개 / 쌍 {sum(len(r["candidates"]) for r in eval_rows)}개')


## 5. 미세조정

**손실이 내려가는지 봅니다.** ADR-057이 적은 선 — *"한 에폭 뒤 학습 손실이 내려가지
않으면 그것이 답이고, 더 돌리지 않습니다."* 그래서 시드의 10%를 검증으로 떼고, 에폭마다
**실측 시간과 VRAM 피크**를 찍습니다(2026-10-20에 어림을 두 번 틀렸습니다).

In [ ]:
import random
import time

import torch
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

_rng = random.Random(20261021)
_seeds = sorted(r['seed'] for r in train_rows)
_rng.shuffle(_seeds)
_cut = max(1, int(len(_seeds) * VAL_RATIO))
val_seeds = set(_seeds[:_cut])

# **시드 단위로 나눕니다.** 쌍 단위로 나누면 같은 시드가 양쪽에 들어가 검증이 샙니다.
def samples_of(rows, keep):
    out = []
    for row in rows:
        if (row['seed'] in val_seeds) != keep:
            continue
        head = train_texts[row['seed']]
        for paper in row['positives']:
            out.append(InputExample(texts=[head, train_texts[paper]], label=1.0))
        for paper in row['negatives']:
            out.append(InputExample(texts=[head, train_texts[paper]], label=0.0))
    return out


train_samples = samples_of(train_rows, keep=False)
val_samples = samples_of(train_rows, keep=True)
_rng.shuffle(train_samples)
print(f'학습 {len(train_samples)}쌍 / 검증 {len(val_samples)}쌍 '
      f'(검증 시드 {len(val_seeds)}개)')

model = CrossEncoder(MODEL, num_labels=1, max_length=MAX_SEQ)

batch = BATCH
while True:
    try:
        loader = DataLoader(train_samples, shuffle=True, batch_size=batch)
        started = time.perf_counter()
        torch.cuda.reset_peak_memory_stats()
        model.fit(
            train_dataloader=loader,
            epochs=EPOCHS,
            warmup_steps=int(len(loader) * EPOCHS * WARMUP_RATIO),
            optimizer_params={'lr': LR},
            show_progress_bar=True,
        )
        break
    except torch.cuda.OutOfMemoryError:
        # **터지는 대신 느려집니다.** 2026-10-20에 배치 256이 OOM이었고 128로 완주했습니다.
        torch.cuda.empty_cache()
        if batch <= 1:
            raise SystemExit('배치 1에서도 OOM입니다. MAX_SEQ를 줄이거나 큰 GPU가 필요합니다.')
        batch //= 2
        print(f'OOM. 배치를 {batch}로 줄여 다시 시작합니다.')

elapsed = time.perf_counter() - started
peak = torch.cuda.max_memory_allocated() / 1e9
print(f'\n학습 끝: {elapsed / 60:.1f}분, 배치 {batch}, VRAM 피크 {peak:.1f}GB')
print(f'  쌍 {len(train_samples)}개 / {EPOCHS}에폭 -> {len(train_samples) * EPOCHS / elapsed:.1f}쌍/초')

### 검증 쌍에서 양성과 음성이 갈리는가

**이것이 "더 돌릴 가치가 있는가"의 신호입니다.** 판정이 아닙니다 — 판정은 저장소가 간선
0개 집단에서 냅니다. 여기서 보는 것은 **학습이 아무 일도 하지 않았는지**뿐입니다.

In [ ]:
import statistics

if val_samples:
    _pairs = [(s.texts[0], s.texts[1]) for s in val_samples]
    _labels = [s.label for s in val_samples]
    _scores = model.predict(_pairs, batch_size=batch, show_progress_bar=True)
    pos = [s for s, y in zip(_scores, _labels) if y == 1.0]
    neg = [s for s, y in zip(_scores, _labels) if y == 0.0]
    if pos and neg:
        print(f'검증 양성 중앙값 {statistics.median(pos):.4f}  '
              f'음성 중앙값 {statistics.median(neg):.4f}')
        # 쌍을 무작위로 하나씩 뽑아 양성이 더 높은 비율(AUC와 같은 양).
        _rng2 = random.Random(7)
        wins = sum(1 for _ in range(20000)
                   if _rng2.choice(pos) > _rng2.choice(neg))
        print(f'양성이 음성보다 높을 확률 ~ {wins / 20000:.3f}  (0.5면 아무 일도 안 한 것)')
    else:
        print('검증에 한쪽 라벨만 있습니다. VAL_RATIO를 올리세요.')
else:
    print('검증 쌍이 없습니다. VAL_RATIO를 올리세요.')

## 6. 평가 쌍 채점 — 저장소가 판정할 수 있는 모양으로

출력 모양을 **사전학습 노트북과 같게** 둡니다. 그래야 저장소가 **같은 명령**으로 판정하고,
같은 후보·같은 정답·같은 실행이 됩니다.

In [ ]:
import re
import time

import torch

_slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-') + '-cocite-ft'
out = Path(f'rerank-{_slug}.jsonl.gz')

scored, done, started = [], 0, time.perf_counter()
batch_eval = batch
with gzip.open(out, 'wt', encoding='utf-8', newline='\n') as handle:
    for index, row in enumerate(eval_rows, 1):
        head = eval_texts[row['seed']]
        pairs_in = [(head, eval_texts[c]) for c in row['candidates']]
        while True:
            try:
                torch.cuda.reset_peak_memory_stats()
                values = model.predict(pairs_in, batch_size=batch_eval)
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if batch_eval <= 1:
                    raise SystemExit('배치 1에서도 OOM입니다.')
                batch_eval //= 2
                print(f'OOM. 채점 배치를 {batch_eval}로 줄입니다.')
        # **모양을 판정기에 맞춥니다.** `evals.related_study.RerankScores`는 줄마다
        # `candidates`(목록)와 `scores`(목록)를 **색인으로 짝지어** 읽고, 수가 다르면
        # 거부합니다. 딕셔너리로 쓰면 KeyError로 죽습니다 - 사전학습 노트북과 같은
        # 모양이어야 같은 명령으로 판정됩니다.
        handle.write(json.dumps({
            'seed': row['seed'],
            'candidates': row['candidates'],
            'scores': [round(float(v), 6) for v in values],
        }, ensure_ascii=False) + '\n')
        scored.append(row['seed'])
        done += len(pairs_in)
        if index % 200 == 0 or index == len(eval_rows):
            took = time.perf_counter() - started
            peak = torch.cuda.max_memory_allocated() / 1e9
            print(f'  {index}/{len(eval_rows)} 시드  {took / 60:.1f}분  '
                  f'{done / took:.1f}쌍/초  배치 {batch_eval}  VRAM 피크 {peak:.1f}GB')

print(f'\n{out.name}  {out.stat().st_size / 1e6:.1f}MB  시드 {len(scored)}개, 쌍 {done}개')


## 7. 저장하고 올리기

In [ ]:
import re
from pathlib import Path

from huggingface_hub import HfApi

# `MODEL`에서 이름을 다시 만듭니다 - 채점이 끝난 뒤 커널이 끊겨도 1·3절과 이 셀만으로
# 올라가야 합니다.
_slug = re.sub(r'[^a-z0-9]+', '-', MODEL.lower()).strip('-') + '-cocite-ft'
_target = Path(f'rerank-{_slug}.jsonl.gz')
if not _target.exists():
    raise SystemExit(f'{_target.name}이 없습니다. 6절을 먼저 돌리세요.')
print(f'{_target.name}  {_target.stat().st_size / 1e6:.1f}MB')

_api = HfApi(token=HF_TOKEN)
_repo = f"{HF_OWNER}/{DATA_REPO}"
_api.create_repo(_repo, repo_type='dataset', private=True, exist_ok=True)
_api.upload_file(path_or_fileobj=str(_target), path_in_repo=_target.name,
                 repo_id=_repo, repo_type='dataset')
print('올림:', _target.name)

if UPLOAD_WEIGHTS:
    # 2.3GB입니다. 다시 학습하지 않으려면 올려 두는 편이 낫고, 점수만 필요하면 끄세요.
    _model_repo = f"{HF_OWNER}/{DATA_REPO}-reranker-cocite-ft"
    model.save(str(Path('finetuned')))
    _api.create_repo(_model_repo, private=True, exist_ok=True)
    _api.upload_folder(folder_path='finetuned', repo_id=_model_repo)
    print('가중치 올림:', _model_repo)

print(f'\nrepo: {_repo}')
print('저장소에서 받아 판정할 명령:')
print(f'  hf download {_repo} {_target.name} \\')
print('      --repo-type dataset --local-dir derived/2026-10-14')
print('  uv run python -m evals.related_study --cases evals/cases_related_future.json \\')
print('      --variants rerank vector \\')
print(f'      --rerank-scores derived/2026-10-14/{_target.name} \\')
print('      --output reports/rerank_cocite_ft.json')